In [1]:
import pandas as pd
import numpy as np
import math

# File paths
job_data_path = r'C:\Users\kipjo\OneDrive\Documents\GitHub\skill-similarity-engine\data\poc\colleague-services\job_data.csv'
job_skill_mapping_path = r'C:\Users\kipjo\OneDrive\Documents\GitHub\skill-similarity-engine\data\poc\colleague-services\job_skill_mapping.csv'
skill_data_path = r'C:\Users\kipjo\OneDrive\Documents\GitHub\skill-similarity-engine\data\poc\colleague-services\skill_data.csv'

# Load data
df_jobs = pd.read_csv(job_data_path)
df_job_skill = pd.read_csv(job_skill_mapping_path)
df_skills = pd.read_csv(skill_data_path)

In [2]:
# Create composite job IDs in the jobs DataFrame
df_jobs['unique_job_id'] = df_jobs['JobID'].astype(str) + '_' + df_jobs['Org Unit Number'].astype(str)

# Merge composite job ID into job-skill mapping
df_job_skill = df_job_skill.merge(
    df_jobs[['JobID', 'Org Unit Number', 'unique_job_id']],
    on='JobID',
    how='left'
)

# Remove rows where Skill_ID is nan, empty, or the string 'nan'
df_job_skill = df_job_skill[
    df_job_skill['Skill_ID'].notna() &
    (df_job_skill['Skill_ID'] != '') &
    (df_job_skill['Skill_ID'].astype(str).str.lower() != 'nan')
]

In [3]:
# Build a mapping: unique_job_id -> set of skill_ids (excluding nan/empty)
job_skills = {}
for _, row in df_job_skill.iterrows():
    job_id = row['unique_job_id']
    skill_id = str(row['Skill_ID']).strip()
    if pd.isna(job_id) or pd.isna(skill_id) or skill_id == '' or skill_id.lower() == 'nan':
        continue  # skip if no org unit mapping or invalid skill
    if job_id not in job_skills:
        job_skills[job_id] = set()
    job_skills[job_id].add(skill_id)

job_ids = list(job_skills.keys())

In [4]:
results = []
for job_from in job_ids:
    skills_from = {s for s in job_skills[job_from] if pd.notna(s) and s != '' and not (isinstance(s, float) and math.isnan(s))}
    for job_to in job_ids:
        if job_from == job_to:
            continue  # skip self-comparison
        skills_to = {s for s in job_skills[job_to] if pd.notna(s) and s != '' and not (isinstance(s, float) and math.isnan(s))}
        intersection = len(skills_from & skills_to)
        coverage = intersection / len(skills_from) if len(skills_from) > 0 else 0.0
        results.append({
            'job_from': job_from,
            'job_to': job_to,
            'coverage': coverage,
            'shared_skills': intersection,
            'total_skills_from': len(skills_from),
            'total_skills_to': len(skills_to)
        })

df_asym = pd.DataFrame(results)
df_asym.head()

In [5]:
# Specify the two jobs to compare
job1 = 'R0002_55043311'
job2 = 'R0043_55035478'

skills1 = {s for s in job_skills[job1] if pd.notna(s) and s != '' and not (isinstance(s, float) and math.isnan(s))}
skills2 = {s for s in job_skills[job2] if pd.notna(s) and s != '' and not (isinstance(s, float) and math.isnan(s))}

coverage_1_in_2 = len(skills1 & skills2) / len(skills1) if len(skills1) > 0 else 0.0
coverage_2_in_1 = len(skills1 & skills2) / len(skills2) if len(skills2) > 0 else 0.0

print(f"Job 1: {job1}")
print(f"Job 2: {job2}")
print(f"Coverage (Job 1's skills in Job 2): {coverage_1_in_2:.2%}")
print(f"Coverage (Job 2's skills in Job 1): {coverage_2_in_1:.2%}")

# Get skill names for each, filtering out nan/empty
skill_names1 = set(df_skills[df_skills['Skill_ID'].isin(skills1)]['Skill_Name'])
skill_names1 = {s for s in skill_names1 if pd.notna(s) and s != '' and not (isinstance(s, float) and math.isnan(s))}

skill_names2 = set(df_skills[df_skills['Skill_ID'].isin(skills2)]['Skill_Name'])
skill_names2 = {s for s in skill_names2 if pd.notna(s) and s != '' and not (isinstance(s, float) and math.isnan(s))}

print(f"\nSkills for {job1}: {skill_names1}")
print(f"\nSkills for {job2}: {skill_names2}")
print(f"\nShared skills: {skill_names1 & skill_names2}")
print(f"\nSkills only in {job1}: {skill_names1 - skill_names2}")
print(f"\nSkills only in {job2}: {skill_names2 - skill_names1}")

In [6]:
print(len(skills1), len(skills2))

print(skills1)

In [7]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,4))
plt.hist(df_asym['coverage'], bins=30, color='seagreen', edgecolor='black', alpha=0.7)
plt.title('Distribution of Asymmetric Skill Coverage Between Jobs')
plt.xlabel('Coverage (from job_from to job_to)')
plt.ylabel('Number of Job Pairs')
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

In [8]:
import pandas as pd
import matplotlib.pyplot as plt

# Path to engine output (update if needed)
engine_path = r'C:\Users\kipjo\OneDrive\Documents\GitHub\skill-similarity-engine\data\poc\output\poc_run_20250528_162415\cross_department_similarities.csv'

# Load engine output
df_engine = pd.read_csv(engine_path)

# Plot distributions
plt.figure(figsize=(10, 5))
plt.hist(
    df_asym['coverage'], bins=30, alpha=0.6, label='Notebook: Asymmetric Coverage',
    color='seagreen', edgecolor='black'
)
plt.hist(
    df_engine['similarity'], bins=30, alpha=0.5, label='Engine: Cross-Department Similarity',
    color='royalblue', edgecolor='black'
)
plt.title('Distribution of Job-to-Job Skill Similarity Scores')
plt.xlabel('Similarity / Coverage')
plt.ylabel('Number of Job Pairs')
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()